# Credit Risk Prediction System
## Business Problem
Lending teams need faster, more consistent risk decisions. This project predicts borrower default probability and translates scores into operational risk segments (Low / Medium / High) to support underwriting and portfolio monitoring.

## Objective
Build a production-grade workflow that converts historical loan records into:
- Default probability estimates
- Risk segmentation
- Feature-level business insights

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_data
from src.preprocessing import preprocess_data
from src.feature_engineering import engineer_features
from src.model_training import train_model
from src.evaluation import evaluate_model
from src.inference import predict_default_probability

## Data Overview

In [ ]:
data_path = PROJECT_ROOT / "data" / "loan.csv"
df_raw = load_data(str(data_path))
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_base = preprocess_data(df_raw)
print(f"Modeling base shape: {df_base.shape}")
df_base[['default', 'annual_inc', 'dti', 'int_rate', 'grade']].head()

## Feature Engineering

In [ ]:
df_features = engineer_features(df_base)
print(f"Feature matrix shape: {df_features.shape}")
df_features.head()

## Model Training

In [ ]:
artifact = train_model(df_features)
evaluation = evaluate_model(artifact, df_features)
evaluation['metrics']

## Model Evaluation

In [ ]:
top_features = evaluation['feature_importance'].head(12)
plt.figure(figsize=(8, 5))
sns.barplot(data=top_features, x='importance', y='feature', palette='crest')
plt.title('Top Feature Importance Drivers')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Business Insights

In [ ]:
X_test = artifact['X_test'].copy()
scored = predict_default_probability(artifact['estimator'], X_test)
segment_summary = scored['risk_segment'].value_counts(normalize=True).mul(100).round(2)
segment_summary

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=scored, x='risk_segment', order=['Low', 'Medium', 'High'], palette=['#2ca25f', '#fdae6b', '#de2d26'])
plt.title('Risk Segment Distribution (Holdout Set)')
plt.xlabel('Risk Segment')
plt.ylabel('Borrower Count')
plt.tight_layout()
plt.show()

## Recommendations
1. Prioritize manual review for High-risk applicants and create tighter policy thresholds for this segment.
2. Use Medium-risk scores for targeted pricing adjustments instead of outright rejection.
3. Monitor feature-importance drift monthly to keep strategy aligned with portfolio behavior.